# Day 2 — NLP: From Text to Numbers
**ComplianceGPT Lab · REU 2026 · Tuesday June 30**

Yesterday you got ~90% accuracy — but only because someone handed you perfectly extracted features.
Today: work directly with raw text. No hand-crafted features.

The progression:
1. **Bag of Words** — ignore word order, just count occurrences
2. **TF-IDF** — weight rare words more, common words less
3. **The semantic gap** — why TF-IDF still can't tell 'court order' from 'judicial mandate'

**No GPU needed. Runs on your laptop.**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics.pairwise import cosine_similarity
print('Imports OK')

In [ ]:
# ── Simulated HIPAA scenario text ────────────────────────────────────────
# Raw text versions of the same 80 scenarios from Day 1
# (In the real pipeline these come from court case documents)

scenarios = [
    ("A federal judge signed a court order requiring the hospital to produce patient records", "PERMITTED"),
    ("The patient signed an authorization form allowing the clinic to share her diagnosis", "PERMITTED"),
    ("A law enforcement officer requested records claiming they were needed for an investigation", "DENIED"),
    ("The hospital disclosed records to the treating physician for continued care", "PERMITTED"),
    ("A researcher obtained a waiver of authorization from the IRB and accessed de-identified data", "PERMITTED"),
    ("The health plan disclosed member records to an employer without authorization", "DENIED"),
    ("A court issued a judicial mandate requiring production of mental health records", "PERMITTED"),
    ("The hospital faxed records to an unverified number without patient consent", "DENIED"),
    ("The patient's attorney subpoenaed records with proper satisfactory assurances on file", "PERMITTED"),
    ("An officer implied there might be a court order and the hospital disclosed records", "DENIED"),
    ("A business associate accessed the database without a signed BAA", "DENIED"),
    ("Treatment records were shared with a specialist under a referral arrangement", "PERMITTED"),
    ("The hospital disclosed records pursuant to a lawful subpoena with patient notification", "PERMITTED"),
    ("Records were sold to a marketing firm without authorization", "DENIED"),
    ("The patient verbally consented and the clinic disclosed to the family member", "DENIED"),  # verbal not sufficient
    ("A court order signed by the presiding judge required immediate production", "PERMITTED"),
    ("The covered entity disclosed to comply with a mandatory reporting statute", "PERMITTED"),
    ("The officer said the records were required and the hospital complied", "DENIED"),
    ("A public health authority requested identifiable records for outbreak investigation", "PERMITTED"),
    ("The employee's supervisor requested medical records without signed authorization", "DENIED"),
] * 4  # repeat to get 80 samples

texts  = [s[0] for s in scenarios]
labels = [s[1] for s in scenarios]

print(f'{len(texts)} scenarios')
print('PERMITTED:', labels.count('PERMITTED'), '  DENIED:', labels.count('DENIED'))

## Step 1 — Bag of Words

The simplest possible text representation: count how many times each word appears.
Word order doesn't matter. 'court order' and 'order court' are identical.
Each scenario becomes a vector with one dimension per unique word in the vocabulary.

In [ ]:
# ── Bag of Words ──────────────────────────────────────────────────────────
bow = CountVectorizer(stop_words='english', max_features=200)
X_bow = bow.fit_transform(texts)

print(f'Vocabulary size: {len(bow.vocabulary_)} words')
print(f'Feature matrix shape: {X_bow.shape}  (80 scenarios × {X_bow.shape[1]} word-counts)')

# Top words
word_counts = np.asarray(X_bow.sum(axis=0)).flatten()
top_words = pd.Series(word_counts, index=bow.get_feature_names_out()).sort_values(ascending=False)[:15]
print('\nTop 15 words:')
print(top_words.to_string())

# Train logistic regression on BoW
lr = LogisticRegression(max_iter=1000, random_state=42)
scores_bow = cross_val_score(lr, X_bow, labels, cv=5, scoring='accuracy')
print(f'\nBag-of-Words accuracy: {scores_bow.mean():.1%} ± {scores_bow.std():.1%}')

## Step 2 — TF-IDF

**TF-IDF = Term Frequency × Inverse Document Frequency**

The idea: 'the', 'a', 'and' appear in *every* document — they're useless for classification.
But 'subpoena' appears in only 3 scenarios — it's highly informative.
TF-IDF automatically downweights common words and upweights rare, informative ones.

In [ ]:
# ── TF-IDF ────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(stop_words='english', max_features=200, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(texts)

print(f'TF-IDF vocabulary: {X_tfidf.shape[1]} terms (including bigrams)')

scores_tfidf = cross_val_score(lr, X_tfidf, labels, cv=5, scoring='accuracy')
print(f'TF-IDF accuracy:    {scores_tfidf.mean():.1%} ± {scores_tfidf.std():.1%}')
print(f'BoW accuracy:       {scores_bow.mean():.1%} ± {scores_bow.std():.1%}')
print(f'Improvement:        {(scores_tfidf.mean() - scores_bow.mean())*100:+.1f} points')

# Most informative features
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42).fit(X_tfidf, labels)
feat_names = tfidf.get_feature_names_out()
coefs = pd.Series(lr_tfidf.coef_[0], index=feat_names).sort_values()
print('\nStrongest DENIED predictors:')
print(coefs.head(5).to_string())
print('\nStrongest PERMITTED predictors:')
print(coefs.tail(5).to_string())

## Step 3 — The Semantic Gap

TF-IDF improved accuracy. But it has a fundamental limitation: it treats every word as independent.
'court order' and 'judicial mandate' mean the same thing legally — but to TF-IDF they're completely unrelated.

Let's measure this directly.

In [ ]:
# ── The semantic gap in TF-IDF space ────────────────────────────────────
phrases = [
    'A court order signed by the judge required production',
    'A judicial mandate from the presiding bench required production',
    'An officer claimed records were needed for the investigation',
    'The hospital disclosed records to the treating physician',
]

phrase_vecs = tfidf.transform(phrases).toarray()
sim = cosine_similarity(phrase_vecs)

print('Cosine similarity between phrases (TF-IDF):')
print('                                               [A]    [B]    [C]    [D]')
labels_p = ['[A] court order         ',
             '[B] judicial mandate    ',
             '[C] officer investigation',
             '[D] treating physician  ']
for i, row in enumerate(sim):
    vals = '  '.join(f'{v:.2f}' for v in row)
    print(f'  {labels_p[i]}  {vals}')

print()
print('TF-IDF thinks:')
print(f'  court order ↔ judicial mandate:  {sim[0,1]:.2f}  (correct label: both PERMITTED)')
print(f'  court order ↔ officer request:   {sim[0,2]:.2f}  (correct label: DENIED vs PERMITTED)')
print()
print('These should be FAR apart but look CLOSE — and CLOSE but look FAR.')
print('TF-IDF has no concept of semantic meaning. It only sees word overlap.')

In [ ]:
# ── YOUR CODE ────────────────────────────────────────────────────────────
# 1. Add 2 new phrases of your own (from the HIPAA scenarios above)
#    that should be semantically similar but TF-IDF would rate as dissimilar
#
# 2. Compute their cosine similarity and print it
#
# 3. Which word in TF-IDF vocabulary would help bridge 'court order' and
#    'judicial mandate'? Print the TF-IDF weight of that word.

# YOUR CODE HERE


**Reflection:**

1. TF-IDF improved over BoW. What specific property of legal text does TF-IDF handle better than BoW?

2. The cosine similarity between 'court order' and 'judicial mandate' was near 0 in TF-IDF space.
   What would need to be different about the representation for them to score high?

3. Think about the adversarial case: 'an officer implied there might be a court order'.
   How would TF-IDF classify this? How would a human classify it? Why the difference?

*Your answers:*

---
**Next:** Day 3 (`day3_embeddings.ipynb`) — word embeddings give words *geometric meaning*.
'court order' and 'judicial mandate' end up close. But 'implied there might be' still trips up the model. That's the LLM problem.